# AGN-Egent — run on your own FITS

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seratsaad/agn-egent/blob/main/notebooks/run_on_your_fits.ipynb)

Upload an AGN spectrum, run the agentic decomposition with **your own Claude or OpenAI
API key**, and get the broad/narrow line separation + single-epoch black-hole mass — for
one object or a batch. Mirrors the Egent workflow.


## 1 · Install (Colab)
Clones AGN-Egent + the PyQSOFit engine and installs dependencies. ~2 min the first time.


In [ ]:
!git clone -q https://github.com/seratsaad/agn-egent.git
%cd agn-egent
!git clone -q https://github.com/legolason/PyQSOFit.git external/PyQSOFit
!pip install -q -e external/PyQSOFit
!pip install -q -r requirements.txt openai


## 2 · Pick a provider and enter your API key
Set `PROVIDER` to `'openai'`, `'claude'`, or `'rule'` (no key / deterministic baseline).
Thread env vars are set **before** importing numpy for reproducible fits.


In [ ]:
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS',
          'VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ[v] = '1'

PROVIDER = 'openai'        # 'openai' | 'claude' | 'rule'
MODEL    = None            # e.g. 'gpt-4o-mini' or 'claude-opus-4-8'; None = default

import getpass
API_KEY = None
if PROVIDER in ('openai', 'claude'):
    API_KEY = getpass.getpass(f'{PROVIDER} API key: ')

import agn_egent
from agn_egent import (load_sdss, load_row_fits, run_agent, run_batch,
                       make_inspector, TriageInspector, derive, render_diagnostic)
print('AGN-Egent ready. reviewer =', PROVIDER)


## 3 · Upload a FITS and decompose
Works on an **SDSS** spec file directly. For a **generic** wavelength/flux/err FITS set
`REDSHIFT` (and `FLUX_SCALE`, e.g. `1e17` for cgs). Outside Colab, set `PATH` manually.


In [ ]:
REDSHIFT   = None     # required only for non-SDSS FITS
FLUX_SCALE = 1.0      # e.g. 1e17 if flux is in erg/s/cm^2/A

try:
    from google.colab import files
    PATH = list(files.upload().keys())[0]
except Exception:
    PATH = 'external/PyQSOFit/example/data/spec-0332-52367-0639.fits'  # demo fallback

try:
    spec = load_sdss(PATH)
except Exception:
    assert REDSHIFT is not None, 'non-SDSS FITS: set REDSHIFT above'
    spec = load_row_fits(PATH, z=REDSHIFT, flux_scale=FLUX_SCALE)
print(spec)


In [ ]:
inspector = make_inspector(PROVIDER, api_key=API_KEY, model=MODEL)
outcome = run_agent(spec, inspector=inspector, finalize_mc=True, nsamp=25, verbose=True)

print(outcome.summary())
print(outcome.final_result.summary())
dq = derive(outcome.final_result, 'Hb')
if dq: print('\nDerived:', dq)


In [ ]:
from IPython.display import Image
Image(render_diagnostic(outcome.final_result, 'diagnostic.png'))


## 4 · Batch
Upload several FITS at once (or build a list of paths). Each runs in its own process; the
reviewer (LLM) is only consulted for the flagged fits. Results come back as a table.


In [ ]:
try:
    from google.colab import files
    paths = list(files.upload().keys())
except Exception:
    d = 'external/PyQSOFit/example/data/'
    paths = [d+'spec-0266-51602-0013.fits', d+'spec-0332-52367-0639.fits']

specs = []
for p in paths:
    try: specs.append(load_sdss(p))
    except Exception as e: print('skip', p, e)

inspector = TriageInspector(make_inspector(PROVIDER, api_key=API_KEY, model=MODEL))
report = run_batch(specs, inspector=inspector, max_workers=2)
print(report.summary())
report.to_csv('results.csv'); report.to_json('results.json')
print('wrote results.csv / results.json')


---
Outputs (figures, `provenance.json`, results tables) are written under `data/runs/`.
See the [repo](https://github.com/seratsaad/agn-egent) and
[showcase](https://seratsaad.github.io/agn-egent) for details.
